# Generate causal EEW v3 after the pilot gate

Preserve all 16,178 original assignments. Store full 100-second raw parents, raw/normalized 1/2/3-second windows, explicit classification/regression eligibility, continuous and sampled arrival metadata, source/parent/background IDs and reproducible seeds. Metadata is separate from waveform channels. Original files are read-only. No model training.


In [1]:
from pathlib import Path
import json,hashlib,sys,time
sys.dont_write_bytecode=True
V3=Path(r"C:/Users/choud/Desktop/Minor project/cnn/cnn wave/eew_causal_v3")
sim_nb=json.loads((V3/"01_causal_simulator_v3.ipynb").read_text(encoding="utf-8"))
# Reuse exact notebook definitions; do not rerun the pilot or any original generation cells.
exec(compile("".join(sim_nb["cells"][1]["source"]),"v3_simulator_definitions","exec"))
gate=readj(V3/"configs/pilot_gate.json")
assert gate["status"]=="PASS"
assert gate["pilot_sha256"]==sha(V3/"reports/v3_counterfactual_results.json")
assert gate["configuration_sha256"]==sha(V3/"configs/simulation_v3.json")
assert readj(V3/"reports/v3_counterfactual_results.json")["simulator_definition_sha256"]==hashlib.sha256(ast.unparse(eq_tree).encode()).hexdigest()
SIM=ROOT/"data wave/synthetic_waveform_simulator/output"
assign=pd.read_csv(SIM/"phase11_preprocessed_dataset/split_indices.csv")
source_meta=pd.read_csv(SIM/"phase2_travel_time/metadata_with_theoretical_arrivals.csv").set_index("source_id")
noise_map=pd.read_csv(SIM/"phase10_classification_dataset/earthquake_noise_mapping.csv").set_index("combined_index")
assert assign.combined_index.is_unique and len(assign)==16178
definition_hash=hashlib.sha256("".join(sim_nb["cells"][1]["source"]).encode()).hexdigest()
print("Pilot gate verified; generating existing assignments only.")


Root causes mapped; fixed gains {'band': 0.7043400785612605, 'pink': 1.3103644084803956} ; original protected files 44
Pilot gate verified; generating existing assignments only.


In [2]:
generation_manifest=[];file_hashes={}
eqgen=CausalEEWGeneratorV3(params);ngen=CausalNoiseWindowGeneratorV3(params)
started=time.perf_counter()
for split,n_expected in [("train",11324),("validation",2426),("test",2428)]:
    destination=V3/"dataset"/f"eew_dataset_{split}.h5"
    done=V3/"configs"/f"{split}_generation_complete.json"
    rows=assign[assign.split==split].reset_index(drop=True)
    assert len(rows)==n_expected
    if done.exists():
        saved=readj(done)
        assert saved["definition_hash"]==definition_hash and sha(destination)==saved["sha256"]
        generation_manifest.extend(saved["records"]);file_hashes[split]=saved["sha256"]
        print(split,"cached verified");continue
    assert not destination.exists(),"Incomplete file present: stop for explicit review, do not overwrite"
    n=len(rows);texttype=h5py.string_dtype("utf-8")
    records=[]
    with h5py.File(destination,"x") as f:
        f.attrs.update(sampling_rate_hz=100.,pre_context_samples=100,split=split,
                       benchmark_type="synthetic oracle-aligned; pending independent verification",
                       simulator_definition_hash=definition_hash,
                       sample_axes="record,time",amplitude_units="relative synthetic amplitude")
        f.create_dataset("parent_raw",(n,10000),dtype="float32",chunks=(32,10000),compression="gzip",compression_opts=1)
        for t,L in [(1,200),(2,300),(3,400)]:
            for kind in ["raw","norm"]:
                f.create_dataset(f"windows_{t}s_{kind}",(n,L),dtype="float32",compression="gzip",compression_opts=1)
            f.create_dataset(f"windows_pure_post_p_{t}s",(n,t*100),dtype="float32",compression="gzip",compression_opts=1)
        numeric={"labels":"int8","magnitude":"float32","depth_km":"float64","hypocentral_distance_km":"float64",
            "source_latitude":"float64","source_longitude":"float64","station_latitude":"float64","station_longitude":"float64",
            "p_arrival_sample":"int32","s_arrival_sample":"int32","reference_sample":"int32",
            "p_arrival_seconds":"float64","s_arrival_seconds":"float64","sp_seconds":"float64",
            "pair_id":"int32","combined_index":"int32","seed":"int64","context_mean":"float64","context_std":"float64",
            "magnitude_mask":"bool"}
        for t in [1,2,3]:
            numeric[f"pre_s_valid_{t}s"]="bool";numeric[f"classification_mask_{t}s"]="bool";numeric[f"magnitude_mask_{t}s"]="bool"
        for k,dtype in numeric.items():f.create_dataset(k,(n,),dtype=dtype)
        string_names=["source_id","trace_name","receiver_code","origin_time_utc","parent_id","background_id",
                      "noise_profile","parent_sha256","background_sha256"]
        for k in string_names:f.create_dataset(k,(n,),dtype=texttype)
        for start in range(0,n,128):
            end=min(n,start+128);size=end-start
            buff={"parent_raw":np.empty((size,10000),np.float32)}
            for t,L in [(1,200),(2,300),(3,400)]:
                for kind in ["raw","norm"]:buff[f"windows_{t}s_{kind}"]=np.empty((size,L),np.float32)
                buff[f"windows_pure_post_p_{t}s"]=np.empty((size,t*100),np.float32)
            vals={k:[] for k in [*numeric,*string_names]}
            for j in range(start,end):
                row=rows.iloc[j];idx=int(row.combined_index);label=int(row.label);pair=int(row.pair_id)
                if label:
                    meta_row=source_meta.loc[str(row.source_id)];seed=42+idx
                    out=eqgen.synthesize_earthquake_window(float(meta_row.Magnitude_Value),float(meta_row.hypocentral_distance_km),
                                                          float(meta_row.theoretical_sp_time_seconds),seed,.05)
                    info={"magnitude":float(meta_row.Magnitude_Value),"depth_km":float(meta_row.Depth_km),
                          "hypocentral_distance_km":float(meta_row.hypocentral_distance_km),
                          "source_latitude":float(meta_row.Lat),"source_longitude":float(meta_row.Long),
                          "station_latitude":float(meta_row.receiver_latitude),"station_longitude":float(meta_row.receiver_longitude),
                          "p_arrival_sample":out["p_sample"],"s_arrival_sample":out["s_sample"],
                          "p_arrival_seconds":out["tp_sec"],"s_arrival_seconds":out["ts_sec"],
                          "sp_seconds":float(meta_row.theoretical_sp_time_seconds),"source_id":str(row.source_id),
                          "trace_name":str(meta_row.trace_name),"receiver_code":str(meta_row.receiver_code),
                          "origin_time_utc":str(meta_row.origin_time_utc),"noise_profile":"gaussian_bandlimited"}
                    bgid=f"eq:{seed}:domain11:role1"
                else:
                    noise_row=noise_map.loc[idx];seed=int(noise_row.seed);profile=str(noise_row.noise_profile_type)
                    out=ngen.generate_noise_window(profile,seed,.05)
                    info={k:-1 for k in ["magnitude","depth_km","hypocentral_distance_km","source_latitude","source_longitude",
                        "station_latitude","station_longitude","p_arrival_sample","s_arrival_sample","p_arrival_seconds","s_arrival_seconds","sp_seconds"]}
                    info.update(source_id=f"NOISE_P{pair}",trace_name=f"NOISE_TRC_{idx}",receiver_code="NOISE_STA",
                                origin_time_utc="",noise_profile=profile)
                    bgid=f"noise:{seed}:domain22:{profile}"
                local=j-start;buff["parent_raw"][local]=out["parent_raw"]
                for t in [1,2,3]:
                    for kind in ["raw","norm"]:buff[f"windows_{t}s_{kind}"][local]=out[f"w{t}_{kind}"]
                    buff[f"windows_pure_post_p_{t}s"][local]=out[f"w{t}_post_raw"]
                    pre=bool(label and out["reference_sample"]+t*100<info["s_arrival_sample"])
                    info[f"pre_s_valid_{t}s"]=pre
                    info[f"classification_mask_{t}s"]=bool(not label or pre)
                    info[f"magnitude_mask_{t}s"]=bool(label and pre)
                info.update(labels=label,pair_id=pair,combined_index=idx,seed=seed,
                    reference_sample=out["reference_sample"],context_mean=out["context_mean"],context_std=out["context_std"],
                    magnitude_mask=bool(label),parent_id=f"v3:{idx}",background_id=bgid,
                    parent_sha256=hashlib.sha256(out["parent_raw"].tobytes()).hexdigest(),
                    background_sha256=hashlib.sha256(out["background"].tobytes()).hexdigest())
                for k in vals:vals[k].append(info[k])
                records.append({"split":split,"row":j,**info})
            for k,array in buff.items():f[k][start:end]=array
            for k,values in vals.items():f[k][start:end]=np.asarray(values,dtype=texttype if k in string_names else numeric[k])
            if start%1024==0:savej(V3/"configs/generation_progress.json",{"split":split,"completed":end,"total":n})
        f.flush()
    digest=sha(destination);file_hashes[split]=digest;generation_manifest.extend(records)
    savej(done,{"definition_hash":definition_hash,"sha256":digest,"records":records})
    print(split,n,"saved; seconds",round(time.perf_counter()-started,1))
pd.DataFrame(generation_manifest).to_csv(V3/"configs/split_manifest.csv",index=False)
savej(V3/"configs/generation_manifest.json",{"status":"COMPLETE","record_count":len(generation_manifest),
      "definition_hash":definition_hash,"simulator_notebook_sha256":sha(V3/"01_causal_simulator_v3.ipynb"),
      "configuration_sha256":sha(V3/"configs/simulation_v3.json"),"files":file_hashes,
      "source_inputs":{str(p.relative_to(ROOT)):sha(p) for p in [SIM/"phase11_preprocessed_dataset/split_indices.csv",
          SIM/"phase2_travel_time/metadata_with_theoretical_arrivals.csv",SIM/"phase10_classification_dataset/earthquake_noise_mapping.csv"]}})
assert all(sha(ROOT/rel)==value for rel,value in before.items())
print("Generation complete; original sources unchanged. Validation still required.")


train 11324 saved; seconds 45.8
validation 2426 saved; seconds 54.4
test 2428 saved; seconds 63.3
Generation complete; original sources unchanged. Validation still required.
